# 看懂 HTTP，手搓 API

## HTTP 的一去一回：请求和响应

HTTP 的一次对话，就是一去一回、两段有固定格式的文本：

- 调用方发过去的叫**请求**（request）。
- 服务方回过来的叫**响应**（response）。

两段报文的结构几乎对称：

~~~text
请求（去）                         响应（回）
├─ 请求行                          ├─ 状态行
├─ 请求头（若干行）                ├─ 响应头（若干行）
├─ （一个空行）                    ├─ （一个空行）
└─ 请求体                          └─ 响应体
~~~

空行非常重要，它是“头部结束、正文开始”的分界线。请求体和响应体不一定存在，但请求行或状态行、头部结束的空行都属于报文格式的一部分。

### 请求的四个部分

#### 1. 请求行

请求行永远只有一行，说明三件事：

~~~text
GET /api/profile HTTP/1.1
~~~

- **方法**：这次请求要做什么，例如 `GET` 是获取数据，`POST` 是提交内容。
- **路径**：要访问对方的哪个资源，例如 `/api/profile`。
- **协议版本**：当前使用的 HTTP 版本，例如 `HTTP/1.1`。

#### 2. 请求头

请求头是一行一条的附加说明，格式统一为“名字: 值”。它可以说明要找哪台服务器、调用方是谁、能接受什么格式，以及请求体是什么格式等。

#### 3. 空行

请求头结束后必须有一个空行。没有这个分界线，接收方就无法可靠判断正文从哪里开始。

#### 4. 请求体

请求体是真正提交的内容，例如 POST 请求提交的 JSON。获取数据的 GET 请求通常没有请求体。

### 响应的四个部分

#### 1. 状态行

响应的第一行叫状态行，说明三件事：

~~~text
HTTP/1.1 200 OK
~~~

- **协议版本**
- **状态码**
- **简短说明**

状态码先记住家族规律：

- `2xx`：请求成功。
- `4xx`：请求方有问题，例如资源不存在、参数不对。
- `5xx`：服务方处理失败。

最常见的两个状态码是：

- `200`：成功。
- `404`：没有找到请求的资源。

#### 2. 响应头

响应头也是一行一条的附加说明。其中最重要的一个是 `Content-Type`，它告诉调用方响应体是什么格式：

- `application/json`：JSON 数据。
- `text/html; charset=utf-8`：HTML 文本，按 UTF-8 解码。
- `text/plain`：普通文本。

#### 3. 空行

响应头结束后同样需要一个空行。

#### 4. 响应体

响应体就是调用方最终“看到”的正文。5.1 中的 `{"ip": ...}`、DeepSeek 返回的 `choices` JSON，都是响应体；上面的状态行和响应头经常会被工具默认隐藏。

## 用 `curl -v` 验证

前面说的是规范，现在让工具把一次完整对话打印出来。`-v` 是 verbose 的缩写，意思是把请求原文（默认不返回请求体）、响应原文和连接过程都显示出来。

~~~bash
curl.exe -v "https://httpbin.org/get"
~~~

输出中先记住行首的三种记号：

- `*`：`curl` 自己的过程说明，例如建立连接、TLS 握手。
- `>`：发出去的请求原文。
- `<`：收到的响应原文。

忽略 `*` 的过程说明后，一次完整的请求和响应大致是这样（具体日期、IP 和服务器名称会因人而异）：

~~~text
> GET /?format=json HTTP/2
> Host: api.ipify.org
> User-Agent: curl/8.7.1
> Accept: */*
>
< HTTP/2 200
< content-type: application/json
< content-length: 22
< server: cloudflare
<
{"ip":"114.86.123.45"}
~~~

这里的 `/?format=json` 是路径加查询参数，`?` 后面的 `format=json` 不属于路径本身。`Host`、`User-Agent`、`Accept` 是请求头，而且是 `curl` 自动带上的。GET 通常没有请求体，所以请求头后的空行后直接开始响应。

### 带请求体的 POST

再看 5.1 中调用大模型的请求。加上 `-v` 后，请求行里的方法会变成 `POST`，并且请求头后面会有请求体：

~~~bash
curl -v 'https://api.deepseek.com/chat/completions' \
  -H 'Content-Type: application/json' \
  -H 'Authorization: Bearer <你的 API Key>' \
  -d '{
        "model": "deepseek-chat",
        "messages": [
          {"role": "user", "content": "你好，请用一句话介绍你自己"}
        ],
        "stream": false
      }'
~~~

要点：

- `-H` 用来手动添加请求头。
- `Content-Type: application/json` 告诉服务端请求体是 JSON。
- `Authorization` 携带身份凭证，真实 API Key 只放在本机环境变量或密钥管理工具中，不要写进笔记、代码仓库或截图。
- `-d` 提交请求正文， `curl` 默认使用 GET ，有请求体默认就是 POST。
- `curl -v` 重点展示报文边界；请求体较长时，终端输出不一定像教程示例一样完整显示正文。

### 常见的方法

| 方法 | 直观含义 | 常见用途 |
| --- | --- | --- |
| `GET` | 把资源给我 | 查询列表、查看详情 |
| `POST` | 我提交内容，请你处理 | 创建资源、提交表单、触发动作 |
| `PUT` | 用这份内容整体替换 | 完整更新一个资源 |
| `PATCH` | 只修改一部分 | 局部更新资源 |
| `DELETE` | 把资源删掉 | 删除资源 |
| `HEAD` | 像 GET，但只要头不要体 | 检查资源是否存在、查看元信息 |
| `OPTIONS` | 我能对这个资源做什么 | 询问服务器支持的方法、CORS 预检 |

实际项目中最常见的是 `GET`、`POST`、`PUT`、`PATCH` 和 `DELETE`。方法名、路径、状态码和请求体共同组成一个 API 的约定。

### 常见的请求头和响应头

请求头：

| 请求头 | 作用 |
| --- | --- |
| `Host` | 要访问哪台服务器 |
| `User-Agent` | 调用方是谁，使用什么工具或浏览器 |
| `Accept` | 调用方能接受什么格式的响应 |
| `Accept-Language` | 调用方偏好的语言 |
| `Content-Type` | 请求体是什么格式 |
| `Content-Length` | 请求体有多少字节 |
| `Authorization` | 身份凭证 |
| `Cookie` | 随请求携带的小数据 |

响应头：

| 响应头 | 作用 |
| --- | --- |
| `Content-Type` | 响应体是什么格式 |
| `Content-Length` | 响应体有多少字节 |
| `Server` | 服务端软件信息 |
| `Date` | 响应生成时间 |
| `Cache-Control` | 缓存策略 |
| `Set-Cookie` | 让浏览器保存 Cookie |
| `Location` | 重定向目标地址 |
| `Access-Control-Allow-Origin` | 允许哪些来源的网页调用我 |

## 手搓 API 要照顾到什么？

为了“返回一段 JSON”，服务端至少要完成这些工作：

1. 持续监听一个端口，等待客户端连接。
2. 识别请求方法和路径。
3. 根据请求方法选择对应的处理逻辑。
4. 返回正确的状态行。
5. 返回必要的响应头，特别是正确的 `Content-Type`。
6. 用空行结束响应头。
7. 把 JSON 序列化成文本，再编码成字节写入响应体。
8. 对不存在的路径返回 `404`。

## 用 Python 实现这套规范

Python 标准库自带 `http.server`，先用它写一个最小 API。新建 `backend/main.py`：

~~~python
from http.server import BaseHTTPRequestHandler, HTTPServer
import json

profile = {
    "heroTitle": "关于我",
    "heroSubtitle": "项目，创意，灵感，我的作品",
}


class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/api/profile":
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()     # 空行
            body = json.dumps(profile, ensure_ascii=False)
            self.wfile.write(body.encode("utf-8"))
        else:
            self.send_response(404)
            self.end_headers()


print("后端已启动：http://localhost:8000/api/profile")
HTTPServer(("", 8000), Handler).serve_forever()       # 持续监听 8000 端口
~~~

这是一个零第三方依赖的后端：`json` 和 `http.server` 都来自 Python 标准库。`ensure_ascii=False` 让中文原样输出。

### 代码和 HTTP 报文的对应关系

| Python 代码 | 它完成的 HTTP 工作 |
| --- | --- |
| `def do_GET(self)` | 接收 GET 请求 |
| `self.path` | 读取请求路径 |
| `self.send_response(200)` | 写入状态行，表示成功 |
| `self.send_header(...)` | 写入响应头 |
| `self.end_headers()` | 结束响应头，并写出头部与正文之间的空行 |
| `json.dumps(profile, ...)` | 把 Python 字典序列化为 JSON 文本 |
| `self.wfile.write(...)` | 把响应体字节写回客户端 |
| `else` 分支 | 对未匹配的路径返回 404 |

`HTTPServer(("", 8000), Handler)` 表示监听 8000 端口，并把请求交给 `Handler` 处理。8000 只是开发中常见的端口号，不是 HTTP 规定的固定端口；正式部署时还会由反向代理、进程管理器等组件负责更多工作。

## 跑起来

进入项目运行服务：

~~~bash
uv run python .\backend\main.py
~~~

程序停在终端并不是卡死，而是在持续监听 8000 端口、等待请求。另开一个终端调用 API：

~~~bash
curl http://localhost:8000/api/profile
~~~

也可以直接在浏览器打开：

~~~text
http://localhost:8000/api/profile
~~~

停止服务使用 `Ctrl + C`。

## 再来一次 `curl -v`：这回是自己的服务器

服务运行后，执行：

~~~bash
curl -v http://localhost:8000/api/profile
~~~

可以看到类似这样的完整报文：

~~~text
> GET /api/profile HTTP/1.1
> Host: localhost:8000
> User-Agent: curl/8.7.1
> Accept: */*
>
< HTTP/1.0 200 OK
< Server: BaseHTTP/0.6 Python/3.x
< Date: ...
< Content-Type: application/json
<
{"heroTitle": "关于我", "heroSubtitle": "项目，创意，灵感，我的作品"}
~~~

逐行对照：

- `> GET /api/profile HTTP/1.1` 是请求行。
- 后面的 `>` 行是调用方的请求头。
- 请求头结束的空行后，服务端发回 `< HTTP/1.0 200 OK` 状态行。
- `< Content-Type: application/json` 是我们在 Python 中写的响应头。
- 最后 JSON 字符串就是响应体。

`http.server` 可能使用 HTTP/1.0 来回复，而客户端之前访问公网服务时可能协商成 HTTP/2。这不影响本节的学习目标：报文的语义和组成仍然是请求行、头、空行、体，以及状态行、头、空行、体。

### 浏览器视角：F12 里的同一份报文

浏览器访问 `http://localhost:8000/api/profile` 后，按 `F12` 打开开发者工具，切到 **Network**，刷新页面并点开这一条请求。

在 **Headers** 中可以看到：

- **General**：请求 URL、请求方法、状态码。
- **Response Headers**：我们通过 `send_header` 写出的响应头。
- **Request Headers**：浏览器发出的请求头。

4.5 中我们用 Network 看过资源加载顺序；这一次点进一个请求的内部，看到的是和 `curl -v` 同一套 HTTP 信息，只是观察角度不同。浏览器把报文解析成图形界面，`curl -v` 则把原文直接打印出来。

## 动手改两处，做两个实验

前面说过，`Content-Type` 是“关于响应体的说明”，服务端也能看到调用方自动带来的请求信息。现在在 `main.py` 中临时改两处，然后重新启动服务：

**改动一：**在 `else` 分支前加入一个 `/hello` 路径：

~~~python
        elif self.path == "/hello":
            self.send_response(200)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.end_headers()
            self.wfile.write("<h1>你好，HTTP</h1>".encode("utf-8"))
~~~

`charset=utf-8` 告诉浏览器响应体中的中文使用 UTF-8 解码。省略它时，浏览器可能猜错编码，页面会出现乱码。

**改动二：**在 `do_GET` 的开头打印请求信息：

~~~python
    def do_GET(self):
        print(self.headers)          # 收到的请求头
        print(self.client_address)   # 请求来自哪个地址
        ...
~~~

保存后按 `Ctrl + C` 停掉旧进程，再重新运行。

### 实验一：响应头的威力

浏览器打开：

~~~text
http://localhost:8000/hello
~~~

当响应头是：

~~~text
Content-Type: text/html; charset=utf-8
~~~

浏览器会把响应体 `<h1>你好，HTTP</h1>` 当作 HTML 渲染成一个大标题。

现在只把代码里的 `text/html` 改成 `text/plain`，重启服务并刷新页面。响应体一个字都没有变，但浏览器会把它当普通文本显示，`<h1>` 标签也会直接露出来。

这个实验说明：

> 内容不变，头一变，对方的处理方式就会改变。

所以，网页和 API 数据在 HTTP 层面没有本质差别，都是响应体；`text/html` 让浏览器按网页渲染，`application/json` 则让调用方按 JSON 解析。

### 实验二：服务端能拿到什么？

先用 `curl` 调用一次：

~~~bash
curl.exe http://localhost:8000/api/profile
~~~

服务端终端通常会打印：

~~~text
Host: localhost:8000
User-Agent: curl/8.21.0
Accept: */*
~~~

再用浏览器访问一次，头部会更多：

~~~text
Host: localhost:8000
User-Agent: Mozilla/5.0 ...
Accept: text/html,application/xhtml+xml,...
Accept-Language: zh-CN,zh;q=0.9
Accept-Encoding: gzip, deflate
...
~~~

添加的两行 print() ：

1. 有些请求头并不是我们在 `main.py` 里写的，而是 `curl` 或浏览器自动带上的。`User-Agent` 让服务端知道调用方是哪个工具或浏览器。
2. `self.client_address` 会打印类似 `('127.0.0.1', 54321)` 的值，其中包含发起请求的 IP 和端口。本机访问本机时，IP 通常是 `127.0.0.1`；换成其他机器访问时，这里会是对方的来源地址。

服务端天然能看到 User-Agent、来源 IP、语言偏好等元信息。这些信息可以用于访问统计、防刷、按语言返回内容，也提醒我们：网络上的请求比自己想象得更“透明”，不要在不信任的网站上随意提交敏感数据。